In [1]:
# First, make sure the installation completes successfully
# Sometimes Jupyter needs a restart after installation
!pip install missingno

# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import scipy.stats as st
import missingno as msno




In [2]:
# function to analyze and clean a dataframe and explain datatypes


import pandas as pd
import numpy as np

def analyze_and_clean_dataframe_explain(df, fill_threshold=0.5, protected_vars=None):
    """
    Analyzes and cleans a pandas DataFrame, dropping columns with low fill rates.
    - Displays fill rate, type, and action for each variable in a table
    - Drops columns with fill rate below the threshold (except protected variables)
    - Provides summary statistics for quantitative and qualitative variables
    - Drops duplicate rows (keeping first occurrence) and reindexes
    - Analyzes dtypes and proposes optimizations for remaining variables

    Parameters:
    df (pd.DataFrame): Input DataFrame
    fill_threshold (float): Threshold for fill rate (default: 0.5).
                           Columns with fill rate below this will be dropped.
    protected_vars (list): List of variable names that should NOT be dropped
                          regardless of fill rate (e.g., ['id', 'customer_key'])

    Returns:
    pd.DataFrame: Cleaned DataFrame with low-fill columns and duplicates removed
    """
    print("="*60)
    print("DATAFRAME ANALYSIS AND CLEANING REPORT")
    print("="*60)

    # 1. Shape of the original dataset
    print(f"\n📊 ORIGINAL DATASET SHAPE: {df.shape[0]} rows × {df.shape[1]} columns")

    # Initialize protected variables list
    if protected_vars is None:
        protected_vars = []

    # 2. Tabular summary of variables, types, and fill rates
    print(f"\n📋 VARIABLE ANALYSIS (Fill Rate Threshold: {fill_threshold})")
    if protected_vars:
        print(f"🛡️  Protected Variables: {protected_vars}")
    print("-" * 80)

    summary_data = []
    variables_to_drop = []

    for column in df.columns:
        dtype = str(df[column].dtype)
        fill_rate = df[column].notna().mean()
        missing_count = df[column].isna().sum()

        # Determine variable type
        if dtype in ["int64", "float64", "int32", "float32"] or pd.api.types.is_numeric_dtype(df[column]):
            var_type = "Quantitative"
        else:
            var_type = "Qualitative"

        # Determine action with protection check
        is_protected = column in protected_vars
        if fill_rate < fill_threshold and not is_protected:
            action = "DROP"
            variables_to_drop.append(column)
        elif fill_rate < fill_threshold and is_protected:
            action = "KEEP (Protected)"
        else:
            action = "KEEP"

        summary_data.append({
            'Variable': column,
            'Type': var_type,
            'Fill Rate': f"{fill_rate:.2%}",
            'Missing': missing_count,
            'Action': action,
            'Protected': '🛡️' if is_protected else ''
        })

    # Create and display summary DataFrame
    summary_df = pd.DataFrame(summary_data)

    # Print formatted table
    print(f"{'Variable':<20} {'Type':<12} {'Fill Rate':<10} {'Missing':<8} {'Protected':<10} {'Action':<15}")
    print("-" * 80)
    for _, row in summary_df.iterrows():
        if row['Action'] == "DROP":
            color = "🔴"
        elif row['Action'] == "KEEP (Protected)":
            color = "🟡"
        else:
            color = "🟢"
        print(f"{row['Variable']:<20} {row['Type']:<12} {row['Fill Rate']:<10} {row['Missing']:<8} {row['Protected']:<10} {color} {row['Action']}")

    # 3. Drop columns with low fill rate
    df_cleaned = df.copy()  # Work with a copy

    if variables_to_drop:
        print(f"\n🗑️  DROPPING COLUMNS: {variables_to_drop}")
        df_cleaned = df_cleaned.drop(columns=variables_to_drop)
        print(f"   Shape after dropping low-fill columns: {df_cleaned.shape}")
    else:
        print(f"\n✅ NO COLUMNS DROPPED (all meet fill rate threshold)")

    # 4. Summary statistics for quantitative variables
    quantitative_cols = df_cleaned.select_dtypes(include=[np.number]).columns

    if len(quantitative_cols) > 0:
        print(f"\n📈 QUANTITATIVE VARIABLES STATISTICS ({len(quantitative_cols)} variables)")
        print("-" * 80)

        stats_df = df_cleaned[quantitative_cols].describe().round(2)

        # Print formatted statistics table
        print(f"{'Statistic':<12}", end="")
        for col in quantitative_cols:
            print(f"{col:<15}", end="")
        print()
        print("-" * (12 + 15 * len(quantitative_cols)))

        for stat in ['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']:
            print(f"{stat:<12}", end="")
            for col in quantitative_cols:
                value = stats_df.loc[stat, col]
                if stat == 'count':
                    print(f"{int(value):<15}", end="")
                else:
                    print(f"{value:<15}", end="")
            print()
    else:
        print(f"\n❌ NO QUANTITATIVE VARIABLES FOUND")

    # 5. Unique values for qualitative variables - ENHANCED TABULAR FORMAT
    qualitative_cols = df_cleaned.select_dtypes(exclude=[np.number]).columns

    if len(qualitative_cols) > 0:
        print(f"\n🏷️  QUALITATIVE VARIABLES ANALYSIS ({len(qualitative_cols)} variables)")
        print("=" * 90)

        # Create tabular data for qualitative variables
        qual_analysis = []
        for col in qualitative_cols:
            unique_count = df_cleaned[col].nunique()
            total_count = len(df_cleaned[col])
            null_count = df_cleaned[col].isnull().sum()
            non_null_count = total_count - null_count

            # Calculate uniqueness ratio
            uniqueness_ratio = unique_count / non_null_count if non_null_count > 0 else 0

            # Determine variable category
            if uniqueness_ratio >= 0.95:
                category = "High Cardinality"
            elif uniqueness_ratio >= 0.5:
                category = "Medium Cardinality"
            elif unique_count <= 10:
                category = "Low Cardinality"
            else:
                category = "Medium Cardinality"

            qual_analysis.append({
                'Variable': col,
                'Unique_Values': unique_count,
                'Total_Count': total_count,
                'Null_Count': null_count,
                'Uniqueness_Ratio': f"{uniqueness_ratio:.2%}",
                'Category': category
            })

        # Print header
        print(f"{'Variable':<20} {'Unique':<8} {'Total':<8} {'Nulls':<8} {'Uniqueness':<12} {'Category':<18}")
        print("-" * 90)

        # Print data
        for item in qual_analysis:
            print(f"{item['Variable']:<20} {item['Unique_Values']:<8} {item['Total_Count']:<8} "
                  f"{item['Null_Count']:<8} {item['Uniqueness_Ratio']:<12} {item['Category']:<18}")

        print("\n" + "=" * 90)

        # Show detailed breakdown for each qualitative variable
        for col in qualitative_cols:
            unique_count = df_cleaned[col].nunique()
            print(f"\n📊 DETAILED ANALYSIS: {col}")
            print("-" * 50)
            print(f"Total unique values: {unique_count}")

            if unique_count == 0:
                print("   → All values are missing")
            elif unique_count == 1:
                print(f"   → Single value: {df_cleaned[col].dropna().iloc[0]}")
            elif 2 <= unique_count <= 15:
                print("   → Value distribution:")
                value_counts = df_cleaned[col].value_counts()
                for idx, (value, count) in enumerate(value_counts.items()):
                    percentage = (count / len(df_cleaned)) * 100
                    print(f"     {idx+1:2d}. '{value}': {count:,} ({percentage:.1f}%)")
            else:
                print("   → Top 5 most frequent values:")
                top_values = df_cleaned[col].value_counts().head(5)
                for idx, (value, count) in enumerate(top_values.items()):
                    percentage = (count / len(df_cleaned)) * 100
                    print(f"     {idx+1}. '{value}': {count:,} ({percentage:.1f}%)")

                if unique_count > 5:
                    print(f"     ... and {unique_count - 5} other unique values")

            # Show null information if present
            null_count = df_cleaned[col].isnull().sum()
            if null_count > 0:
                null_percentage = (null_count / len(df_cleaned)) * 100
                print(f"   → Missing values: {null_count:,} ({null_percentage:.1f}%)")
    else:
        print(f"\n❌ NO QUALITATIVE VARIABLES FOUND")

    # 6. Handle duplicates
    num_duplicates = df_cleaned.duplicated().sum()
    print(f"\n🔍 DUPLICATE ANALYSIS")
    print("-" * 30)
    print(f"Number of duplicate rows: {num_duplicates}")

    if num_duplicates > 0:
        # Drop duplicates keeping the first occurrence
        df_final = df_cleaned.drop_duplicates(keep='first').reset_index(drop=True)
        print(f"✅ Removed {num_duplicates} duplicate rows (kept first occurrence)")
    else:
        df_final = df_cleaned.reset_index(drop=True)
        print("✅ No duplicates found")

    print(f"Final shape: {df_final.shape}")

    # 7. Dtype Analysis and Optimization Proposals
    print(f"\n🔍 DTYPE ANALYSIS AND OPTIMIZATION PROPOSALS")
    print("=" * 60)

    dtype_proposals = []
    for col in df_final.columns:
        dtype = df_final[col].dtype
        unique_count = df_final[col].nunique()
        total_count = len(df_final[col])
        null_count = df_final[col].isnull().sum()
        non_null_count = total_count - null_count

        # Propose dtype optimization
        if pd.api.types.is_integer_dtype(dtype):
            min_val = df_final[col].min()
            max_val = df_final[col].max()
            if min_val >= 0 and max_val <= 255:
                proposal = "uint8"
            elif min_val >= 0 and max_val <= 65535:
                proposal = "uint16"
            elif min_val >= 0 and max_val <= 4294967295:
                proposal = "uint32"
            elif min_val >= -128 and max_val <= 127:
                proposal = "int8"
            elif min_val >= -32768 and max_val <= 32767:
                proposal = "int16"
            elif min_val >= -2147483648 and max_val <= 2147483647:
                proposal = "int32"
            else:
                proposal = "int64"
        elif pd.api.types.is_float_dtype(dtype):
            if df_final[col].dropna().between(-3.4e38, 3.4e38).all():
                proposal = "float32"
            else:
                proposal = "float64"
        elif pd.api.types.is_object_dtype(dtype):
            if df_final[col].dropna().apply(lambda x: isinstance(x, str)).all():
                if unique_count / non_null_count < 0.5:
                    proposal = "category"
                else:
                    proposal = "object"
            else:
                proposal = "object"
        elif pd.api.types.is_bool_dtype(dtype):
            proposal = "bool"
        else:
            proposal = str(dtype)

        dtype_proposals.append({
            'Variable': col,
            'Current_dtype': str(dtype),
            'Unique_Values': unique_count,
            'Null_Count': null_count,
            'Optimized_dtype': proposal,
            'Memory_Savings': f"{(df_final[col].memory_usage(deep=True) - df_final[col].astype(proposal).memory_usage(deep=True)) / 1024:.1f} KB"
        })

    # Print dtype proposals
    print(f"{'Variable':<20} {'Current_dtype':<15} {'Unique':<8} {'Nulls':<8} {'Optimized_dtype':<15} {'Memory_Savings':<15}")
    print("-" * 80)
    for item in dtype_proposals:
        print(f"{item['Variable']:<20} {item['Current_dtype']:<15} {item['Unique_Values']:<8} "
              f"{item['Null_Count']:<8} {item['Optimized_dtype']:<15} {item['Memory_Savings']:<15}")

    # Summary
    print(f"\n📊 CLEANING SUMMARY")
    print("-" * 30)
    print(f"Original shape: {df.shape}")
    print(f"Columns dropped: {len(variables_to_drop)}")
    print(f"Duplicates removed: {num_duplicates}")
    print(f"Final shape: {df_final.shape}")
    print(f"Data retention: {(df_final.shape[0] * df_final.shape[1]) / (df.shape[0] * df.shape[1]):.1%}")

    return df_final


In [3]:
# Function to trim the entire data frame


import pandas as pd

def trim_dataframe(df):
    """
    Trims leading/trailing whitespace from column names and string data in a DataFrame.

    Parameters:
    df (pd.DataFrame): Input DataFrame

    Returns:
    pd.DataFrame: DataFrame with trimmed column names and string data
    """
    # Trim column names
    df.columns = df.columns.str.strip()

    # Trim string data in all object-type columns
    for col in df.select_dtypes(include=['object']).columns:
        df[col] = df[col].str.strip()

    return df


In [4]:
# Function to filter on valid countries

import pycountry
import pandas as pd

def filter_valid_country_codes(df, country_code_col='Country Code'):
    """
    Filters a DataFrame to keep only rows with valid 3-letter country codes (ISO Alpha-3)
    using the `pycountry` package.

    Parameters:
    - df (pd.DataFrame): Input DataFrame.
    - country_code_col (str): Name of the column containing country codes (default: 'Country Code').

    Returns:
    - pd.DataFrame: Filtered DataFrame with only valid country codes.
    - list: List of invalid country codes that were filtered out.
    """
    # Get all valid 3-letter country codes from pycountry
    valid_codes = {country.alpha_3 for country in pycountry.countries}

    # Get unique country codes from the DataFrame
    unique_codes = df[country_code_col].drop_duplicates().dropna()

    # Identify invalid codes (not in pycountry's list)
    invalid_codes = unique_codes[~unique_codes.isin(valid_codes)].unique()

    # Filter DataFrame to keep only valid country codes
    filtered_df = df[~df[country_code_col].isin(invalid_codes)]

       # Print results
    print("Invalid country codes filtered out:", invalid_codes)
    print("Shape after filtering:", filtered_df.shape)

    return filtered_df, invalid_codes


ModuleNotFoundError: No module named 'pycountry'

The table gives information about the series code (indicators) and their characteristics.


Doesn't seem to be very important as data. I will only keep series code, topic and Long definition. 

# TEST


In [ ]:

EdStatsSeries = pd.read_csv('EdStatsSeries.csv')
EdStatsSeries.head()




In [ ]:
EdStatsSeries['Indicator Name'].nunique()

In [ ]:
EdStatsSeries = EdStatsSeries[EdStatsSeries['Indicator Name'].isin( [
    "Projection: Population age 15-19 in thousands by highest level of educational attainment. Post Secondary. Total",
    "Projection: Population age 15-19 in thousands by highest level of educational attainment. Upper Secondary. Total",
    "GDP (constant 2010 US$)",
    "GDP (current US$)",
    "GDP per capita (current US$)",
    "Government expenditure on secondary education as % of GDP (%)",
    "Government expenditure on tertiary education as % of GDP (%)",
    "Government expenditure per secondary student (US$)",
    "Government expenditure per secondary student as % of GDP per capita (%)",
    "Government expenditure per tertiary student (US$)",
    "Government expenditure per tertiary student as % of GDP per capita (%)",
    "Internet users (per 100 people)",
    "Personal computers (per 100 people)",
    "Adjusted net enrolment rate, lower secondary, both sexes (%)",
    "Adjusted net enrolment rate, upper secondary, both sexes (%)",
    "Enrolment in secondary general, both sexes (number)",
    "Gross enrolment ratio, secondary, both sexes (%)",
    "Enrolment in tertiary education, all programmes, both sexes (number)"
])]
# the indicators are chosen based on the needs to understand secondary and tertiary education and their expenditures

EdStatsSeries['Indicator Name'].nunique()

In [ ]:
# Keep columns for the analysis
columns_to_keep = ['Series Code', 'Indicator Name']
EdStatsSeries = EdStatsSeries[columns_to_keep]

EdStatsSeries.head()

EdStatsCountry

Information related to the countries in the project.

In [ ]:
%%capture
EdStatsCountry = pd.read_csv('EdStatsCountry.csv')
EdStatsCountry.head().T

In [ ]:
%%capture
EdStatsCountry = analyze_and_clean_dataframe_explain(EdStatsCountry, fill_threshold=0.4)  # Adjust threshold as needed

# TEST2

In [ ]:
%%capture
EdStatsCountry['Income Group'] = EdStatsCountry['Income Group'].astype('category')

EdStatsCountry = trim_dataframe(EdStatsCountry)

# Remove Invalid Country codes
EdStatsCountry, invalid_codes = filter_valid_country_codes(EdStatsCountry, country_code_col='Country Code')

EdStatsCountry.head()

In [ ]:
# Keep columns which are useful for the analysis
columns_to_keep = ['Country Code', 'Short Name', 'Income Group', 'Source of most recent Income and expenditure data']
EdStatsCountry = EdStatsCountry[columns_to_keep]
EdStatsCountry.head()


EdStatsCountrySeries

Link between countries and series code (indicator in the first table)

In [ ]:
%%capture
EdStatsCountrySeries = pd.read_csv('EdStatsCountrySeries.csv')
EdStatsCountrySeries.head()

In [ ]:
%%capture
EdStatsCountrySeries = analyze_and_clean_dataframe_explain(EdStatsCountrySeries, fill_threshold=0.5)  # Adjust threshold as needed

In [ ]:
EdStatsCountrySeries = trim_dataframe(EdStatsCountrySeries)

EdStatsCountrySeries = EdStatsCountrySeries.rename(columns={'CountryCode': 'Country Code', 'SeriesCode': 'Series Code'})


# Remove Invalid Country codes
EdStatsCountrySeries, invalid_codes = filter_valid_country_codes(EdStatsCountrySeries, country_code_col='Country Code')

EdStatsCountrySeries['Country Indicator Conc'] = EdStatsCountrySeries['Country Code'] + EdStatsCountrySeries['Series Code']


EdStatsCountrySeries.head()

EdStatsData

Link between country name and indicator codes with years and in a pivot format. shouldbe melted down. 

In [ ]:
EdStatsData = pd.read_csv('EdStatsData.csv')
EdStatsData.head().T

In [ ]:
EdStatsData = pd.melt(EdStatsData, 
                    id_vars=['Country Name','Country Code','Indicator Name','Indicator Code'], 
                    value_vars= None,
                    var_name='Year', 
                    value_name= 'Value')

EdStatsData.head()


In [ ]:

EdStatsData = EdStatsData[ ~EdStatsData['Value'].isnull() ]

EdStatsData.head()

In [ ]:
EdStatsData = analyze_and_clean_dataframe_explain(EdStatsData, fill_threshold=0.5)  # Adjust threshold as needed

In [ ]:
EdStatsData, invalid_codes = filter_valid_country_codes(EdStatsData, country_code_col='Country Code')

In [ ]:
EdStatsData['Country Indicator Conc'] = EdStatsData['Country Code'] + EdStatsData['Indicator Code']
EdStatsData.head()

EdStatsFootNote

Link between country code and the series code

In [ ]:
EdStatsFootNote = pd.read_csv('EdStatsFootNote.csv')
EdStatsFootNote.head()

In [ ]:
EdStatsFootNote = analyze_and_clean_dataframe(EdStatsFootNote, fill_threshold=0.5)  # Adjust threshold as needed

In [ ]:
EdStatsFootNote.head()

In [ ]:
# Check the filtered out countries
CountryList = EdStatsFootNote[['CountryCode']].drop_duplicates(keep='first').reset_index(drop=True)

# Apply the function
CountryList['is_country'] = CountryList['CountryCode'].apply(is_valid_country_code)

# Show non-countries
non_countries = CountryList[~CountryList['is_country']]['CountryCode'].unique()  # Added missing closing quote after 'Name'

# filter country list on non-countries
filtered_CountryList = CountryList[CountryList["CountryCode"].isin(non_countries)]

filtered_CountryList

In [ ]:
EdStatsFootNote = EdStatsFootNote[~EdStatsFootNote["CountryCode"].isin(non_countries)]
EdStatsFootNote.shape

In [ ]:

EdStatsFootNote['Country Indicator Conc'] = EdStatsFootNote['CountryCode'] + EdStatsFootNote['SeriesCode']
EdStatsFootNote['Year'] = EdStatsFootNote['Year'].str[2:]
EdStatsFootNote.head()


PERFORM MERGES OF ALL TABLES

EdStatsFootNote has been completely processed
EdStatsCountrySeries has been completely processed
EdStatsCountry has been completey processed
EdStatsSeries has been completely processed



In [ ]:
MERGED_TABLE = pd.merge(EdStatsData, EdStatsFootNote, on='Country Indicator Conc', how='left')

# Keep only essential columns
essential_columns = [
    'Country Name', 'Country Code', 'Indicator Name', 
    'Indicator Code', 'Year_x','Value', 'Country Indicator Conc', 'DESCRIPTION'
]

MERGED_TABLE = MERGED_TABLE[essential_columns].rename(columns={
 
    'DESCRIPTION': 'IndicatorDescription',
    'Year_x': 'Year',
    
})

MERGED_TABLE.head()

In [ ]:



MERGED_TABLE2 = pd.merge(MERGED_TABLE, EdStatsCountrySeries, on='Country Indicator Conc', how='left')

MERGED_TABLE2 = MERGED_TABLE2.drop(['CountryCode', 'SeriesCode'], axis=1)

MERGED_TABLE2.head()


In [ ]:


MERGED_TABLE3 = pd.merge(MERGED_TABLE2, EdStatsCountry, on='Country Code', how='left')

MERGED_TABLE3 = MERGED_TABLE3.drop(['Short Name'], axis=1)
MERGED_TABLE3.head()

In [ ]:
# Check column names in both DataFrames
print("MERGED_TABLE3 columns:")
print(MERGED_TABLE3.columns.tolist())

EdStatsSeries = EdStatsSeries.rename(columns={'Series Code': 'Indicator Code'})

print("\nEdStatsSeries columns:")
print(EdStatsSeries.columns.tolist())

In [ ]:
MERGED_TABLE4 = pd.merge(
    MERGED_TABLE3,
    EdStatsSeries,
    on='Indicator Code',
    how='left',
    suffixes=('', '_dup')
)

MERGED_TABLE4.head()

In [ ]:
print(MERGED_TABLE4.columns)

In [ ]:
MERGED_TABLE4 = MERGED_TABLE4.drop(columns=['Indicator Name_dup'])

In [ ]:
print(MERGED_TABLE4.columns)

In [ ]:
MERGED_TABLE4.head()

In [ ]:
CONSOLIDATED = MERGED_TABLE4

In [ ]:
CONSOLIDATED ['Indicator Code'].value_counts()